# 03 PyCaret Modeling

? notebook??? cleaned dataset? ???? ??? ??? ???? supervised machine learning ??? ????.

## 1. Library import

In [ ]:
# Run only if needed
# !pip install pycaret imbalanced-learn

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pycaret.classification import *

## 2. Load cleaned data

In [ ]:
data_path = Path('processed_data/knhanes_young_hypertension_2017_2023.csv')

df = pd.read_csv(data_path)
print(df.shape)
df.head()

## 3. Select target and features

HE_sbp? HE_dbp? hypertension target? ?? ? ???? ??? feature?? ????. ??? data leakage? ??? ?????.

In [ ]:
features = [
    'age', 'sex',
    'HE_BMI', 'HE_glu', 'HE_TG',
    'sm_presnt', 'dr_month', 'pa_aerobic',
    'incm', 'educ', 'covid_period'
]

target = 'hypertension'

model_df = df[features + [target]].copy()
print(model_df.shape)
model_df.head()

In [ ]:
# Check missing values before modeling
model_df.isna().sum().sort_values(ascending=False)

In [ ]:
# Target imbalance check
model_df[target].value_counts(normalize=True)

## 4. PyCaret setup

PyCaret?? ???? preprocessing? ????.

- categorical encoding
- missing value handling
- normalization/scaling
- class imbalance handling

In [ ]:
cat_features = ['sex', 'sm_presnt', 'dr_month', 'pa_aerobic', 'incm', 'educ', 'covid_period']
num_features = ['age', 'HE_BMI', 'HE_glu', 'HE_TG']

clf = setup(
    data=model_df,
    target=target,
    categorical_features=cat_features,
    numeric_features=num_features,
    session_id=123,
    train_size=0.8,
    normalize=True,
    fix_imbalance=True,
    fold=5,
    verbose=True
)

## 5. Compare models by AUC

In [ ]:
top3_models = compare_models(sort='AUC', n_select=3)
top3_models

In [ ]:
# Save comparison table
compare_results = pull()
compare_results

## 6. Hyperparameter tuning for top 3 models

In [ ]:
tuned_models = []

for model in top3_models:
    tuned = tune_model(model, optimize='AUC')
    tuned_models.append(tuned)

In [ ]:
# Tuned model results from the last tuning step
pull()

## 7. Evaluate top 3 models

? model? ?? confusion matrix, ROC curve, feature importance? ????.

In [ ]:
for i, model in enumerate(tuned_models, start=1):
    print('Model', i, model)
    plot_model(model, plot='confusion_matrix')
    plot_model(model, plot='auc')
    try:
        plot_model(model, plot='feature')
    except Exception as e:
        print('Feature importance plot is not available for this model:', e)

## 8. Final model

In [ ]:
# Choose the best tuned model based on AUC and final evaluation
best_model = tuned_models[0]

final_model = finalize_model(best_model)
final_model

In [ ]:
predictions = predict_model(final_model)
predictions.head()

In [ ]:
# Save final model for later use
save_model(final_model, 'processed_data/final_hypertension_model')

## 9. Short interpretation

- AUC? ???? ??? ????.
- SBP/DBP? target ???? ???? feature??? ????.
- ??? class? ??? fix_imbalance=True? ???? imbalance ??? ????.